In [7]:
"""
UFA 2012-2025: Multi-Season Analysis Dashboard
==============================================

Generates 8 PNG figures for the GitHub README, summarizing the analysis of
13 seasons of the Ultimate Frisbee Association (formerly AUDL).

Pipeline:
    1. Load player game-stats CSV (one row per player-season)
    2. Filter players with >=4 games played
    3. Reconstruct season totals from per-game averages
    4. Compute the "Impact Score" composite metric
    5. Aggregate stats at the team-season level
    6. Render 8 figures into ./figs/

Figures:
    01 - League overview (titles per franchise + champions timeline)
    02 - Players with most titles
    03 - Career GOATs (offense, defense, longevity)
    04 - Statistical profile of champions (z-scores)
    05 - Elite depth (champions vs non-champions)
    06 - Year N -> Year N+1 prediction
    07 - Regular season vs playoff
    08 - League-wide evolution over time

Usage:
    python ufa_dashboard.py

Requires: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats as ss
from sklearn.linear_model import LinearRegression


# ============================================================
# CONFIG
# ============================================================
# Path to the consolidated dataset (output of consolidate.py).
DATA_PATH = "ufa_player_stats_all_years.csv"

# Output directory where PNG figures are written.
OUT_DIR = Path("figs")
OUT_DIR.mkdir(exist_ok=True)

# Color palette used consistently across all figures.
# - champ:   reserved for the champion / "stand out" category
# - other:   non-champion / neutral baseline
# - primary: primary highlight (offense, positive z-score)
# - accent:  secondary highlight (defense, negative z-score)
# - success: tertiary, used for completion %
PALETTE = {
    "champ":   "#D62728",
    "other":   "#7F7F7F",
    "primary": "#1F77B4",
    "accent":  "#FF7F0E",
    "success": "#2CA02C",
}

# Global seaborn / matplotlib theme. Applied to every figure unless
# overridden inside a specific plot function.
sns.set_theme(
    style="whitegrid",
    rc={
        "axes.edgecolor": "0.3",
        "grid.color": "0.92",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
        "savefig.dpi": 150,
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
    },
)


# ============================================================
# LEAGUE CONSTANTS
# ============================================================
# UFA / AUDL champions by year. Tags match the `team_primary` column
# in the dataset. Source: WatchUFA + Wikipedia. Note that 2020 is missing
# (season cancelled due to COVID-19).
CHAMPS = {
    2012: "PHS", 2013: "TOR", 2014: "SJ",  2015: "SJ",  2016: "DAL",
    2017: "SF",  2018: "MAD", 2019: "NY",  2021: "RAL", 2022: "NY",
    2023: "NY",  2024: "MIN", 2025: "BOS",
}

# Full name lookup for the team tags above (used in figure annotations).
CHAMP_NAMES = {
    "PHS": "Philadelphia Spinners", "TOR": "Toronto Rush",
    "SJ":  "San Jose Spiders",      "DAL": "Dallas Roughnecks",
    "SF":  "SF FlameThrowers",      "MAD": "Madison Radicals",
    "NY":  "New York Empire",       "RAL": "Raleigh Flyers",
    "MIN": "Minnesota Wind Chill",  "BOS": "Boston Glory",
}

# Franchise consolidation: maps relocated/rebranded team tags to a single
# canonical franchise name. E.g. San Jose Spiders -> Oakland Spiders is
# the same franchise. Used in fig01 to count titles per franchise.
FRANCHISE = {
    "SJ":  "Bay Area",  "OAK": "Bay Area",
    "RAL": "Carolina",  "CAR": "Carolina",
    "POR": "Oregon",    "ORE": "Oregon",
}

# EXACT regular-season standing of the UFA champion in each year, validated
# year-by-year against WatchUFA + Wikipedia + Ultiworld.
# Format: year -> (league_rank, division_rank, n_teams_in_league)
# Ties in win-loss record are broken by playoff seeding (division position).
# IMPORTANT: this is the corrected version. An earlier draft used a
# plus/minus proxy that biased against low-volume teams (e.g. Boston 2025).
CHAMP_REGULAR_RANK = {
    2012: (1, 1, 8),
    2013: (1, 1, 12),
    2014: (1, 1, 17),
    2015: (3, 2, 25),
    2016: (1, 1, 26),
    2017: (2, 1, 24),
    2018: (1, 1, 23),
    2019: (3, 2, 21),
    2021: (2, 1, 22),
    2022: (1, 1, 25),
    2023: (3, 2, 24),
    2024: (3, 2, 24),
    2025: (3, 2, 24),
}


# ============================================================
# DATA LOADING & FEATURE ENGINEERING
# ============================================================
def load_data(path):
    """Load the consolidated CSV and add derived columns.

    Adds:
        - team_primary: first team tag (some players were traded mid-season,
          listed as "TEAM1, TEAM2"; we keep only the first one for simplicity).
        - franchise: canonical franchise after consolidating rebrandings.
        - is_champion_team: boolean flag for rows belonging to that year's champion.
    """
    df = pd.read_csv(path)
    df["team_primary"] = df["teams"].str.split(",").str[0].str.strip()
    df["franchise"] = df["team_primary"].map(FRANCHISE).fillna(df["team_primary"])
    df["is_champion_team"] = df.apply(
        lambda r: CHAMPS.get(r["year"]) == r["team_primary"], axis=1
    )
    return df


def add_totals_and_impact(df_q):
    """Reconstruct season totals from per-game averages and compute Impact Score.

    The CSV stores PER-GAME averages (e.g. scores=4.21 means 4.21 scores per
    game on average). We multiply by gamesPlayed to recover season totals,
    which are easier to compare across players with different game counts.

    Impact Score formula (composite metric balancing offense and defense):
        Impact = scores
               + 1.5 * blocks       (blocks are rarer than scores)
               + 2.0 * callahans    (very rare and devastating event)
               - 1.0 * throwaways   (giving the disc back is expensive)
               - 0.5 * drops        (less costly than throwaways)
               - 0.5 * stalls       (situational)

    Then multiplied by a completion-percentage penalty:
        - if completion >= 90%: full credit (penalty = 1.0)
        - if completion <  90%: penalty = compl_pct / 90  (linear scaling)

    NaN completion percentages are filled with the dataset median to avoid
    dropping rows for what is usually a missing-by-design column in early years.
    """
    counted_cols = [
        "scores", "assists", "goals", "blocks", "throwaways", "drops",
        "callahans", "plusMinus", "completions", "stalls",
    ]
    for c in counted_cols:
        df_q[f"{c}_tot"] = df_q[c] * df_q["gamesPlayed"]

    df_q["impact_szn"] = (
        df_q["scores_tot"]
        + 1.5 * df_q["blocks_tot"]
        + 2.0 * df_q["callahans_tot"]
        - 1.0 * df_q["throwaways_tot"]
        - 0.5 * df_q["drops_tot"]
        - 0.5 * df_q["stalls_tot"]
    )
    median_compl = df_q["completionPercentage"].median()
    df_q["compl_pct_filled"] = df_q["completionPercentage"].fillna(median_compl)
    df_q["compl_penalty"] = np.where(
        df_q["compl_pct_filled"] >= 90, 1.0, df_q["compl_pct_filled"] / 90
    )
    df_q["impact_szn_adj"] = df_q["impact_szn"] * df_q["compl_penalty"]

    # League-wide percentile rank within each season (used for "top 10%" cutoffs).
    df_q["impact_pct_year"] = df_q.groupby("year")["impact_szn_adj"].rank(pct=True)
    return df_q


def aggregate_team_year(df):
    """Aggregate per-game team-level stats by (year, team).

    For each team-year, computes:
        - games_team: number of games the team played (max gamesPlayed of any
          player on the roster — proxy for team's total games).
        - <stat>_pg: team total of the stat divided by games_team.
        - compl_pct: team-level completion percentage, computed correctly
          from totals (NOT averaging player-level percentages, which would be
          biased toward low-volume players).
    """
    rows = []
    for (year, team), g in df.groupby(["year", "team_primary"]):
        games = g["gamesPlayed"].max()
        if games == 0:
            continue
        out = {"year": year, "team_primary": team, "games_team": games}
        for c in ["scores", "assists", "goals", "blocks", "throwaways", "drops"]:
            out[f"{c}_pg"] = (g[c] * g["gamesPlayed"]).sum() / games

        # Team completion %: total successful completions divided by total
        # offensive possessions (completions + throwaways + stalls).
        compl_t = (g["completions"] * g["gamesPlayed"]).sum()
        throw_t = (g["throwaways"] * g["gamesPlayed"]).sum()
        stall_t = (g["stalls"] * g["gamesPlayed"]).sum()
        denom = compl_t + throw_t + stall_t
        out["compl_pct"] = compl_t / denom if denom > 0 else np.nan
        rows.append(out)
    return pd.DataFrame(rows)


# ============================================================
# FIGURE 1 — LEAGUE OVERVIEW
# ============================================================
def fig01_overview(out_path):
    """Two-panel overview: titles per franchise + champions timeline.

    Left panel: horizontal bar chart of titles per franchise. Multi-title
    franchises (NY Empire, Bay Area Spiders) are highlighted in red.
    Right panel: chronological timeline, one bar per season, colored by franchise.
    """
    champ_df = pd.DataFrame(
        [(y, t, FRANCHISE.get(t, t), CHAMP_NAMES.get(t, t)) for y, t in CHAMPS.items()],
        columns=["year", "tag", "franchise", "team_name"],
    )
    titles_per = (
        champ_df.groupby("franchise").size().reset_index(name="titles")
        .sort_values("titles", ascending=False)
    )

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.4]}
    )

    # Left panel: titles by franchise
    sns.barplot(
        data=titles_per, x="titles", y="franchise",
        hue="franchise",
        palette=[PALETTE["champ"] if t >= 2 else PALETTE["primary"]
                 for t in titles_per["titles"]],
        ax=ax1, edgecolor="white", linewidth=1.5, legend=False,
    )
    ax1.set_xlabel("UFA Titles")
    ax1.set_ylabel("")
    ax1.set_title("Titles by Franchise (2012-2025)")
    ax1.set_xticks(range(0, titles_per["titles"].max() + 1))
    for i, v in enumerate(titles_per["titles"]):
        ax1.text(v + 0.05, i, str(v), va="center", fontweight="bold")

    # Right panel: chronological champions timeline
    champ_df_sorted = champ_df.sort_values("year").reset_index(drop=True)
    franchise_order = sorted(set(champ_df["franchise"]))
    palette_map = dict(zip(franchise_order,
                           sns.color_palette("tab10", len(franchise_order))))

    sns.barplot(
        data=champ_df_sorted.assign(bar_value=1),
        x="bar_value", y="year",
        hue="franchise", palette=palette_map,
        orient="h", ax=ax2, edgecolor="white", linewidth=2,
        dodge=False, legend=False,
    )
    for i, row in champ_df_sorted.iterrows():
        ax2.text(0.5, i, f"{row['year']} — {row['team_name']}",
                 va="center", ha="center", fontweight="bold",
                 color="white", fontsize=10)
    ax2.set_xticks([])
    ax2.set_yticks([])
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("")
    ax2.set_ylabel("")
    ax2.set_title("Champions Timeline")
    for s in ("left", "bottom"):
        ax2.spines[s].set_visible(False)

    plt.suptitle("UFA 2012-2025: League Overview",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path / "01_overview.png")
    plt.close()


# ============================================================
# FIGURE 2 — PLAYERS WITH MOST TITLES
# ============================================================
def fig02_titles_per_player(df, out_path):
    """Top 15 players ranked by number of UFA titles.

    A player "has a title" in year Y if they played >=4 games for the
    champion of year Y. The 4-game threshold avoids counting late-season
    callups or bench warmers who barely played.
    """
    rows = []
    for y, t in CHAMPS.items():
        roster = df[(df["year"] == y) & (df["team_primary"] == t) &
                    (df["gamesPlayed"] >= 4)]
        for _, r in roster.iterrows():
            rows.append({"year": y, "team": t,
                         "playerID": r["playerID"], "name": r["name"]})
    appearances = pd.DataFrame(rows)
    top = (
        appearances.groupby(["playerID", "name"])
        .agg(titles=("year", "nunique"),
             years=("year", lambda x: sorted(x.tolist())),
             teams=("team", lambda x: "/".join(sorted(set(x)))))
        .reset_index()
        .sort_values("titles", ascending=False)
        .head(15)
    )

    # Color code: 4+ titles (red), 3 (orange), 2 or fewer (blue)
    top["color_group"] = pd.cut(
        top["titles"], bins=[0, 2, 3, 99],
        labels=["2-", "3", "4+"],
    ).astype(str)
    palette = {"4+": PALETTE["champ"], "3": PALETTE["accent"], "2-": PALETTE["primary"]}

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(
        data=top, x="titles", y="name",
        hue="color_group", palette=palette,
        dodge=False, edgecolor="white", linewidth=1.5, ax=ax, legend=False,
    )
    ax.set_xlabel("UFA Titles")
    ax.set_ylabel("")
    ax.set_title("Top 15 Players With Most UFA Titles (2012-2025)")
    ax.set_xticks(range(0, int(top["titles"].max()) + 2))
    for i, r in enumerate(top.itertuples()):
        annotation = f"{r.titles}  ·  {r.teams}  ·  {r.years}"
        ax.text(r.titles + 0.07, i, annotation, va="center", fontsize=9, color="0.3")
    ax.set_xlim(0, top["titles"].max() + 2.5)

    plt.figtext(
        0.05, -0.02,
        "Beau Kittredge holds the all-time record: 5 titles across 4 different franchises.",
        fontsize=10, style="italic", color="0.3",
    )
    plt.tight_layout()
    plt.savefig(out_path / "02_titles_per_player.png")
    plt.close()


# ============================================================
# FIGURE 3 — CAREER GOATS
# ============================================================
def fig03_career_goats(df_q, out_path):
    """Three-panel career leaderboards: offense, defense, longevity.

    Career stats are summed across all seasons. Players with fewer than
    30 career games are excluded (cuts noise from short careers / cup-of-coffee
    appearances). This filter is conservative — most legends have 80+ games.
    """
    career = (
        df_q.groupby("playerID")
        .agg(name=("name", "first"),
             seasons=("year", "nunique"),
             total_games=("gamesPlayed", "sum"),
             scores_career=("scores_tot", "sum"),
             blocks_career=("blocks_tot", "sum"))
        .reset_index()
    )
    career_q = career[career["total_games"] >= 30]

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))

    # Each panel: (column to rank by, x-axis label, panel title, bar color)
    panels = [
        ("scores_career", "Career total scores",
         "Offensive GOAT (Goals + Assists)", PALETTE["primary"]),
        ("blocks_career", "Career total blocks",
         "Defensive GOAT (Blocks)", PALETTE["accent"]),
        ("seasons", "Seasons played",
         "Longevity", PALETTE["champ"]),
    ]
    for ax, (col, xlabel, title, color) in zip(axes, panels):
        # For longevity, tie-break by total_games so prolific veterans rank above
        # part-time long-tenured players.
        if col == "seasons":
            top = career_q.sort_values(
                ["seasons", "total_games"], ascending=[False, False]
            ).head(12)
        else:
            top = career_q.nlargest(12, col)

        sns.barplot(
            data=top, x=col, y="name",
            color=color, edgecolor="white", linewidth=1.5, ax=ax,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel("")
        ax.set_title(title)
        for i, r in enumerate(top.itertuples()):
            v = getattr(r, col)
            if col == "seasons":
                label = f"{int(v)} ({int(r.total_games)} games)"
                offset = 0.15
            else:
                label = f"{int(v)}"
                offset = top[col].max() * 0.01
            ax.text(v + offset, i, label, va="center", fontsize=9)
        ax.set_xlim(0, top[col].max() * 1.18)

    plt.suptitle("UFA Career GOATs (min. 30 games)",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path / "03_career_goats.png")
    plt.close()


# ============================================================
# FIGURE 4 — STATISTICAL PROFILE OF CHAMPIONS
# ============================================================
def fig04_champion_profile(team_yr, out_path):
    """Two-panel champion profile analysis.

    Left panel: heatmap of champion z-scores per metric, year-by-year.
        z-score = (champion's metric - mean of non-champions) / std of non-champions
        Positive z = champion above league average; negative = below.

    Right panel: mean z-score across all 13 champions, by metric.
        Reveals which metrics the champion CONSISTENTLY outperforms in
        (positive bars) and which it consistently underperforms in (negative bars).
        The annotation also shows the % of years where the champion was above
        average — a stricter test of consistency than the mean alone.
    """
    metric_cols   = ["scores_pg", "blocks_pg", "throwaways_pg", "drops_pg", "compl_pct"]
    metric_labels = ["Scores",    "Blocks",    "Throwaways",     "Drops",     "Completion %"]

    # Build z-score matrix: rows = years, columns = metrics.
    rows = []
    for year, champ_team in CHAMPS.items():
        year_data = team_yr[team_yr["year"] == year]
        champ = year_data[year_data["team_primary"] == champ_team]
        others = year_data[year_data["team_primary"] != champ_team]
        if len(champ) == 0 or len(others) < 3:
            continue
        row = {"year": year}
        for m in metric_cols:
            std = others[m].std()
            row[m] = ((champ[m].iloc[0] - others[m].mean()) / std) if std > 0 else 0
        rows.append(row)
    z_df = pd.DataFrame(rows).set_index("year")[metric_cols]
    z_df.columns = metric_labels

    fig, axes = plt.subplots(
        1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1, 1.2]}
    )

    # Left panel: heatmap
    sns.heatmap(
        z_df, annot=True, fmt=".1f", cmap="RdBu_r", center=0,
        cbar_kws={"label": "Z-score (champion vs league)"},
        linewidths=0.5, linecolor="white", ax=axes[0],
        vmin=-2.5, vmax=2.5,
    )
    axes[0].set_title("Champion z-score by metric and year")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Title year")

    # Right panel: aggregate consistency across all 13 seasons
    consistency = pd.DataFrame([
        {"metric": lbl,
         "z_mean": z_df[lbl].mean(),
         "pct_above": (z_df[lbl] > 0).mean() * 100}
        for lbl in metric_labels
    ]).sort_values("z_mean")

    sns.barplot(
        data=consistency, x="z_mean", y="metric",
        hue="metric",
        palette=[PALETTE["primary"] if z > 0 else PALETTE["accent"]
                 for z in consistency["z_mean"]],
        edgecolor="white", linewidth=1.5, ax=axes[1], legend=False,
    )
    axes[1].axvline(0, color="0.3", linewidth=1)
    axes[1].set_xlabel("MEAN z-score of champion (vs. rest of league)")
    axes[1].set_ylabel("")
    axes[1].set_title("Consistent pattern: what do champions have in common?")

    # Annotations are always anchored to the right of zero, regardless of bar
    # direction. This avoids overlap with y-axis labels for negative bars.
    for i, r in enumerate(consistency.itertuples()):
        txt = f"{r.z_mean:+.2f}  ({r.pct_above:.0f}% above average)"
        if r.z_mean > 0:
            axes[1].text(r.z_mean + 0.05, i, txt,
                         va="center", ha="left", fontsize=9)
        else:
            axes[1].text(0.05, i, txt,
                         va="center", ha="left", fontsize=9, color="0.3")
    axes[1].set_xlim(-1.7, 2.2)

    plt.suptitle("Statistical profile of the 13 UFA champions",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path / "04_champion_profile.png")
    plt.close()


# ============================================================
# FIGURE 5 — ELITE DEPTH (CHAMPIONS vs NON-CHAMPIONS)
# ============================================================
def fig05_elite_depth(df_q, out_path):
    """Champion teams have more top-10% players than non-champion teams.

    "Top 10%" is computed using the Impact Score percentile within each season,
    so the threshold adjusts as the league grows (8 teams in 2012 vs 24 in 2025).

    Left panel: boxplot of non-champions, plus stripplot of the 13 champion
    points individually. The strip+box combo is necessary because n=13 makes
    a pure boxplot for champions look like a flat line.

    Right panel: per-year line chart showing where the champion fell relative
    to the league max and league mean.
    """
    # Mark each player as top 10% or not (within their season)
    df_q["top10pct_impact"] = (df_q["impact_pct_year"] >= 0.90).astype(int)

    # Aggregate to team-year: count top-10% players per team
    team_depth = (
        df_q.groupby(["year", "team_primary"])
        .agg(n_top10pct=("top10pct_impact", "sum"),
             team_total_impact=("impact_szn_adj", "sum"))
        .reset_index()
    )
    team_depth["is_champ"] = team_depth.apply(
        lambda r: CHAMPS.get(r["year"]) == r["team_primary"], axis=1
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # --- Left panel: boxplot + stripplot ---
    plot_df = team_depth.copy()
    plot_df["status"] = plot_df["is_champ"].map(
        {True: "Champions", False: "Non-champions"}
    )
    counts = plot_df["status"].value_counts()
    nc_lbl = f"Non-champions\n(n={counts['Non-champions']})"
    c_lbl  = f"Champions\n(n={counts['Champions']})"
    plot_df["status_lbl"] = plot_df["status"].apply(
        lambda s: c_lbl if s == "Champions" else nc_lbl
    )

    sns.boxplot(
        data=plot_df, x="status_lbl", y="n_top10pct",
        hue="status_lbl",
        order=[nc_lbl, c_lbl],
        palette={nc_lbl: PALETTE["other"], c_lbl: PALETTE["champ"]},
        ax=axes[0], width=0.5, linewidth=1.5,
        medianprops=dict(color="white", linewidth=2),
        flierprops=dict(marker="o", markerfacecolor="0.5", markersize=4),
        legend=False,
    )
    # Overlay individual champion points (n=13 — small enough to plot every one)
    sns.stripplot(
        data=plot_df[plot_df["status"] == "Champions"],
        x="status_lbl", y="n_top10pct",
        order=[nc_lbl, c_lbl],
        color="white", edgecolor=PALETTE["champ"], linewidth=1.5,
        size=8, jitter=0.08, ax=axes[0], zorder=10,
    )
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Players in league top 10% (Impact)")
    axes[0].set_title("Elite depth: champions ~5, non-champions ~2.6")

    means = plot_df.groupby("status")["n_top10pct"].mean()
    axes[0].axhline(means["Champions"], color=PALETTE["champ"],
                    linestyle="--", alpha=0.6,
                    label=f"Champions mean: {means['Champions']:.1f}")
    axes[0].axhline(means["Non-champions"], color=PALETTE["other"],
                    linestyle="--", alpha=0.6,
                    label=f"Non-champions mean: {means['Non-champions']:.1f}")
    axes[0].legend(loc="lower right")

    # --- Right panel: per-year timeline ---
    yearly = (
        team_depth.groupby("year")["n_top10pct"]
        .agg(["max", "mean"]).reset_index()
        .rename(columns={"max": "League max", "mean": "League mean"})
    )
    yearly_long = yearly.melt(id_vars="year", var_name="aggregate",
                               value_name="n_top10pct")

    sns.lineplot(
        data=yearly_long, x="year", y="n_top10pct", hue="aggregate",
        style="aggregate",
        markers={"League max": "o", "League mean": "s"},
        dashes=False,
        palette={"League max": "0.6", "League mean": PALETTE["other"]},
        markersize=8, linewidth=1.8, ax=axes[1],
    )
    champ_data = team_depth[team_depth["is_champ"]].sort_values("year")
    sns.lineplot(
        data=champ_data, x="year", y="n_top10pct",
        marker="D", markersize=10, linewidth=2.5,
        color=PALETTE["champ"], label="Champion", ax=axes[1], zorder=5,
    )
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Top 10% players (Impact)")
    axes[1].set_title("Elite depth per year: champion vs league")
    axes[1].legend(loc="upper left", title="")
    axes[1].set_xticks([2012, 2014, 2016, 2018, 2021, 2023, 2025])

    plt.suptitle("Talent concentration: champion teams have more depth",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path / "05_elite_depth.png")
    plt.close()


# ============================================================
# FIGURE 6 — YEAR-OVER-YEAR PREDICTION (N -> N+1)
# ============================================================
def fig06_year_over_year(df_q, out_path):
    """How well does year N performance predict year N+1?

    Left panel: scatter of scores/game in year N vs year N+1, with linear
    regression overlay. Shows persistence (positive slope) but also large noise.

    Right panel: regression to the mean. Players are bucketed by their year-N
    percentile; the bars compare their average percentile in year N vs N+1.
    Top buckets (95-100%) drop ~9 points the next year — they remain elite
    but rarely repeat their peak. Bottom bucket (<50%) regresses upward.
    """
    df_q["scores_total"] = df_q["scores"] * df_q["gamesPlayed"]

    # Build pairs of consecutive years (skipping 2020 which is missing)
    years = sorted(df_q["year"].unique())
    pairs = [(y, y + 1) for y in years if y + 1 in years]

    # Inner join each consecutive year on playerID (player must have played both)
    panel_rows = []
    for y0, y1 in pairs:
        a = df_q[df_q["year"] == y0][[
            "playerID", "name", "scores", "scores_total", "gamesPlayed"
        ]].rename(columns={
            "scores": "scores_n", "scores_total": "scores_total_n",
            "gamesPlayed": "gP_n",
        })
        b = df_q[df_q["year"] == y1][["playerID", "scores", "scores_total"]].rename(
            columns={"scores": "scores_n1", "scores_total": "scores_total_n1"}
        )
        merged = a.merge(b, on="playerID", how="inner")
        merged["year_n"] = y0
        panel_rows.append(merged)
    panel = pd.concat(panel_rows, ignore_index=True)

    r, _ = ss.pearsonr(panel["scores_n"], panel["scores_n1"])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # --- Left panel: scatter + regression ---
    sns.regplot(
        data=panel, x="scores_n", y="scores_n1",
        scatter_kws={"alpha": 0.3, "s": 15, "color": PALETTE["primary"],
                     "edgecolor": "none"},
        line_kws={"color": PALETTE["champ"], "linewidth": 2},
        ax=axes[0], ci=None,
    )
    max_v = panel["scores_n"].max()
    axes[0].plot([0, max_v], [0, max_v], "--", color="0.5", linewidth=1,
                 label="x = y (no change)")
    axes[0].set_xlabel("Scores/game in year N")
    axes[0].set_ylabel("Scores/game in year N+1")
    axes[0].set_title(
        f"Year-over-year persistence  ·  r = {r:.3f}  ·  R² = {r ** 2:.2f}"
    )
    axes[0].legend(loc="upper left")

    # --- Right panel: regression to the mean ---
    # Compute percentile within year for both N and N+1
    panel["pct_n"] = panel.groupby("year_n")["scores_n"].rank(pct=True)
    panel_n1 = panel[["playerID", "year_n", "scores_n1"]].copy()
    panel_n1["year_n1"] = panel_n1["year_n"] + 1
    panel_n1["pct_n1"] = panel_n1.groupby("year_n1")["scores_n1"].rank(pct=True)
    panel = panel.merge(
        panel_n1[["playerID", "year_n", "pct_n1"]], on=["playerID", "year_n"]
    )
    panel["bucket"] = pd.cut(
        panel["pct_n"], bins=[0, 0.5, 0.75, 0.9, 0.95, 1.0],
        labels=["<50%", "50-75%", "75-90%", "90-95%", "95-100%"],
    )

    bucket_long = (
        panel.groupby("bucket", observed=True)
        .agg(N=("pct_n", "mean"), **{"N+1": ("pct_n1", "mean")})
        .reset_index()
        .melt(id_vars="bucket", var_name="year_label", value_name="percentile")
    )
    bucket_long["percentile"] *= 100

    sns.barplot(
        data=bucket_long, x="bucket", y="percentile", hue="year_label",
        palette={"N": PALETTE["primary"], "N+1": PALETTE["accent"]},
        edgecolor="white", linewidth=1.2, ax=axes[1],
    )
    axes[1].set_xlabel("Year-N percentile bucket (scores/game)")
    axes[1].set_ylabel("Mean percentile (%)")
    axes[1].set_title("Regression to the mean: top 5% drops ~9 pp the next year")
    axes[1].legend(title="Year", loc="upper left")
    axes[1].set_ylim(0, 110)

    # Delta annotations on top of the N+1 bars
    deltas = (
        panel.groupby("bucket", observed=True)
        .agg(n_mean=("pct_n", "mean"), n1_mean=("pct_n1", "mean"))
        .reset_index()
    )
    for i, r_ in enumerate(deltas.itertuples()):
        diff = (r_.n1_mean - r_.n_mean) * 100
        axes[1].text(i, r_.n1_mean * 100 + 12,
                     f"{diff:+.1f}pp", ha="center", fontsize=9, color="0.3")

    plt.suptitle(f"Year-over-year prediction (n = {len(panel)} player-year pairs)",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path / "06_year_over_year.png")
    plt.close()


# ============================================================
# FIGURE 7 — REGULAR SEASON vs PLAYOFF
# ============================================================
def fig07_regular_vs_playoff(out_path):
    """Did the regular-season #1 win the playoff?

    Uses CHAMP_REGULAR_RANK, which contains EXACT regular-season standings
    validated against external sources (WatchUFA + Wikipedia + Ultiworld).
    DOES NOT use a plus/minus proxy (an earlier draft did, and it biased
    against low-volume teams like Boston 2025).

    Left panel: histogram of where the champion finished in the regular season.
    Right panel: comparison of early era (2012-2018) vs modern era (2019-2025)
    in terms of how often the #1 team won the title.
    """
    rows = []
    for year, (rk_league, rk_div, n) in CHAMP_REGULAR_RANK.items():
        rows.append({
            "year": year,
            "rank_league": rk_league,
            "rank_div": rk_div,
            "n_teams": n,
            "era": "2012-2018" if year <= 2018 else "2019-2025",
            "champ": CHAMPS[year],
        })
    df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # --- Left panel: distribution of champion's league rank ---
    rank_dist = df["rank_league"].value_counts().sort_index().reset_index()
    rank_dist.columns = ["rank", "count"]
    rank_dist["color_grp"] = rank_dist["rank"].apply(
        lambda r: "champ" if r == 1 else ("primary" if r <= 3 else "accent")
    )

    sns.barplot(
        data=rank_dist, x="rank", y="count",
        hue="color_grp",
        palette={"champ": PALETTE["champ"], "primary": PALETTE["primary"],
                 "accent": PALETTE["accent"]},
        dodge=False, edgecolor="white", linewidth=1.5, ax=axes[0], legend=False,
    )
    axes[0].set_xlabel("Champion's regular-season rank (official W-L)")
    axes[0].set_ylabel("Occurrences")
    axes[0].set_title("Where did the champion finish in the regular season?")
    total = rank_dist["count"].sum()
    for i, r in enumerate(rank_dist.itertuples()):
        pct = r.count / total * 100
        axes[0].text(i, r.count + 0.1, f"{r.count}\n({pct:.0f}%)",
                     ha="center", fontsize=10, fontweight="bold")
    axes[0].set_ylim(0, rank_dist["count"].max() + 1.5)

    # --- Right panel: early era vs modern era ---
    era_stats = (
        df.groupby("era")
        .agg(total=("year", "count"),
             rank1=("rank_league", lambda x: (x == 1).sum()))
        .reset_index()
    )
    era_stats["pct_top1"] = era_stats["rank1"] / era_stats["total"] * 100
    era_stats["label"] = era_stats.apply(
        lambda r: f"Era\n{r['era']}\n(n={r['total']})", axis=1
    )

    sns.barplot(
        data=era_stats, x="label", y="pct_top1",
        hue="era",
        palette={"2012-2018": PALETTE["primary"], "2019-2025": PALETTE["champ"]},
        edgecolor="white", linewidth=1.5, width=0.6, ax=axes[1], legend=False,
    )
    axes[1].set_xlabel("")
    axes[1].set_ylabel("% of years where #1 won the playoff")
    axes[1].set_title("The league changed: early era vs modern era")
    axes[1].set_ylim(0, 100)
    for i, r in enumerate(era_stats.itertuples()):
        axes[1].text(i, r.pct_top1 + 2,
                     f"{r.pct_top1:.0f}%\n({r.rank1}/{r.total})",
                     ha="center", fontsize=11, fontweight="bold")

    plt.suptitle(
        "Regular Season vs Playoff: the best of the regular doesn't always win",
        fontsize=15, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.savefig(out_path / "07_regular_vs_playoff.png")
    plt.close()


# ============================================================
# FIGURE 8 — LEAGUE-WIDE EVOLUTION OVER TIME
# ============================================================
def fig08_league_evolution(team_yr, out_path):
    """4-panel grid showing how the league has evolved over the years.

    Each panel shows a season-level metric (averaged across all teams)
    plotted year-by-year, with a linear trend line overlaid.

    Notable patterns:
      - Goals/game and blocks/game declining (offense more efficient)
      - Throwaways/game dropped sharply in 2019
      - Completion % rising steadily (more careful possession game)
    """
    # Average each team-stat across all teams in each year, then melt to
    # long format for seaborn faceting.
    yearly = (
        team_yr.groupby("year")
        .agg(goals=("goals_pg", "mean"),
             blocks=("blocks_pg", "mean"),
             throwaways=("throwaways_pg", "mean"),
             completion=("compl_pct", "mean"))
        .reset_index()
        .melt(id_vars="year", var_name="metric", value_name="value")
    )
    rename_map = {
        "goals":      "Goals/game (league avg)",
        "blocks":     "Blocks/game",
        "throwaways": "Throwaways/game",
        "completion": "Completion %",
    }
    yearly["metric_label"] = yearly["metric"].map(rename_map)
    color_map = {
        "Goals/game (league avg)": PALETTE["primary"],
        "Blocks/game":             PALETTE["accent"],
        "Throwaways/game":         PALETTE["champ"],
        "Completion %":            PALETTE["success"],
    }

    g = sns.relplot(
        data=yearly, x="year", y="value", col="metric_label",
        hue="metric_label", palette=color_map,
        col_wrap=2, kind="line", marker="o",
        markersize=8, linewidth=2,
        facet_kws={"sharey": False}, height=4, aspect=1.6,
        legend=False,
    )
    g.set_titles("{col_name}")

    # Overlay a linear trend line on each subplot.
    for ax in g.axes.flat:
        title = ax.get_title()
        sub = yearly[yearly["metric_label"] == title]
        if len(sub) >= 2:
            z = np.polyfit(sub["year"], sub["value"], 1)
            xs = np.array([sub["year"].min(), sub["year"].max()])
            ax.plot(xs, np.polyval(z, xs), "--", color="0.5",
                    linewidth=1.2, alpha=0.7)
        ax.set_xticks([2012, 2014, 2016, 2018, 2021, 2023, 2025])
        ax.set_xlabel("Year")
        ax.set_ylabel("")
        if title == "Completion %":
            ax.yaxis.set_major_formatter(
                plt.FuncFormatter(lambda v, _: f"{v:.0%}")
            )

    g.figure.suptitle("League evolution: 2012-2025 trends",
                      fontsize=15, fontweight="bold", y=1.03)
    g.figure.savefig(out_path / "08_league_evolution.png", bbox_inches="tight")
    plt.close(g.figure)


# ============================================================
# MAIN ENTRY POINT
# ============================================================
def main():
    """Load data, compute features, render all 8 figures."""
    print("Loading data...")
    df = load_data(DATA_PATH)

    # Filter players with at least 4 games (avoids small-sample noise)
    df_q = df[df["gamesPlayed"] >= 4].copy()
    df_q = add_totals_and_impact(df_q)

    print("Aggregating team-year stats...")
    team_yr = aggregate_team_year(df)

    # Each entry: (label printed during run, callable that produces the figure)
    figs = [
        ("[1/8] League overview",            lambda: fig01_overview(OUT_DIR)),
        ("[2/8] Titles per player",          lambda: fig02_titles_per_player(df, OUT_DIR)),
        ("[3/8] Career GOATs",               lambda: fig03_career_goats(df_q, OUT_DIR)),
        ("[4/8] Champion profile",           lambda: fig04_champion_profile(team_yr, OUT_DIR)),
        ("[5/8] Elite depth",                lambda: fig05_elite_depth(df_q, OUT_DIR)),
        ("[6/8] Year-over-year prediction",  lambda: fig06_year_over_year(df_q, OUT_DIR)),
        ("[7/8] Regular vs Playoff",         lambda: fig07_regular_vs_playoff(OUT_DIR)),
        ("[8/8] League evolution",           lambda: fig08_league_evolution(team_yr, OUT_DIR)),
    ]
    for label, fn in figs:
        print(label)
        fn()

    print("\n" + "=" * 60)
    print(f"Done — figures saved to {OUT_DIR.resolve()}")
    print("=" * 60)
    for f in sorted(OUT_DIR.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.0f} KB)")


if __name__ == "__main__":
    main()

Loading data...
Aggregating team-year stats...
[1/8] League overview
[2/8] Titles per player
[3/8] Career GOATs
[4/8] Champion profile
[5/8] Elite depth
[6/8] Year-over-year prediction
[7/8] Regular vs Playoff
[8/8] League evolution

Done — figures saved to C:\Users\vinic\OneDrive\Documentos\0-Python Scripts\Exemplo\figs
  01_overview.png  (106 KB)
  01_visao_geral.png  (107 KB)
  02_titles_per_player.png  (124 KB)
  02_titulos_por_jogador.png  (124 KB)
  03_career_goats.png  (167 KB)
  03_goats_carreira.png  (165 KB)
  04_champion_profile.png  (175 KB)
  04_perfil_campeoes.png  (172 KB)
  05_depth_elite.png  (149 KB)
  05_elite_depth.png  (142 KB)
  06_predicao_n_n1.png  (311 KB)
  06_year_over_year.png  (313 KB)
  07_regular_vs_playoff.png  (85 KB)
  08_evolucao_liga.png  (186 KB)
  08_league_evolution.png  (185 KB)
